# Multi-Task Learning in PyTorch: When Gradients Conflict

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/conflicting_gradients_multitask.ipynb)

Companion notebook to [Multi-Task Learning in PyTorch: When Gradients Conflict](https://sesen.ai/blog/multi-task-learning-pytorch-conflicting-gradients).

Two tasks share one small encoder. You will measure the angle between their
gradients, implement PCGrad, and run the 2x2 that separates the problem PCGrad
solves from the one it does not. CPU only, a couple of minutes end to end.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(2)
D_IN, N_SAMPLES, HIDDEN = 8, 512, 6
ADAM_LR = 1e-3      # from the optimisers post: https://sesen.ai/blog/optimisers-sgd-momentum-adam-from-scratch
STEPS = 3000

## 1. Two tasks on one input matrix

Both tasks use the same functional form. The only thing that changes between the
two pairs is which input dimensions task B reads, so any difference in behaviour
comes from that and not from one task being harder than the other.

`amp_b` scales task B's target, which scales its loss and therefore its gradient.

In [ ]:
def make_tasks(pair, seed=0, amp_b=3.0):
    g = torch.Generator().manual_seed(1234 + seed)
    x = torch.rand(N_SAMPLES, D_IN, generator=g) * 2 - 1

    def target(i, j):
        return torch.sin(2.5 * x[:, i] + 1.5 * x[:, j]) + 0.5 * x[:, i] * x[:, j]

    y_a = target(0, 1)
    y_b = target(2, 3) if pair == "conflict" else target(0, 1)
    return x, y_a.unsqueeze(1), (amp_b * y_b).unsqueeze(1)


class SharedTrunk(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(D_IN, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.head_a = nn.Linear(hidden, 1)
        self.head_b = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.trunk(x)
        return self.head_a(h), self.head_b(h)


print("shared trunk parameters:", sum(p.numel() for p in SharedTrunk().trunk.parameters()))

## 2. Measuring the conflict

The usual step calls `(loss_a + loss_b).backward()`. To measure an angle you need
the two gradients separately, so take them with `torch.autograd.grad` and flatten
each into one vector.

In [ ]:
def flat_grad(loss, params, retain=True):
    grads = torch.autograd.grad(loss, params, retain_graph=retain, allow_unused=True)
    return torch.cat([(torch.zeros_like(p) if g is None else g).reshape(-1)
                      for g, p in zip(grads, params)])

## 3. PCGrad

Yu et al. (2020), Algorithm 1, for two tasks. When the gradients disagree, each
one gives up the part of itself that points against the other. When they agree,
this returns the plain sum, so the method is a no-op outside the conflicting case.

In [ ]:
def pcgrad_combine(g_a, g_b):
    dot = torch.dot(g_a, g_b)
    if dot >= 0:
        return g_a + g_b
    pc_a = g_a - dot / g_b.dot(g_b) * g_b
    pc_b = g_b - dot / g_a.dot(g_a) * g_a
    return pc_a + pc_b

## 4. The training loop

The combined vector goes onto the shared trunk only. Each head receives its own
task's gradient, because a head that only one task can see has nothing to conflict
with.

Alongside the losses the loop records three diagnostics per step: the cosine, the
ratio of the two gradient norms, and whether the summed update would increase task
A's loss (`g_A . (g_A + g_B) < 0`).

In [ ]:
def assign_flat(params, vec):
    i = 0
    for p in params:
        n = p.numel()
        p.grad = vec[i:i + n].view_as(p).clone()
        i += n


def train(method, pair, seed=0, steps=STEPS, weight_b=1.0, amp_b=3.0):
    torch.manual_seed(seed)
    x, y_a, y_b = make_tasks(pair, seed=seed, amp_b=amp_b)
    model = SharedTrunk()
    opt = torch.optim.Adam(model.parameters(), lr=ADAM_LR)
    trunk_params = list(model.trunk.parameters())
    head_params = list(model.head_a.parameters()) + list(model.head_b.parameters())
    var_a, var_b = float(y_a.var(unbiased=False)), float(y_b.var(unbiased=False))
    w_b = weight_b if method == "weighted" else 1.0

    hist = {k: [] for k in ("nmse_a", "nmse_b", "cos", "ratio", "back_a")}
    for _ in range(steps):
        pred_a, pred_b = model(x)
        loss_a = ((pred_a - y_a) ** 2).mean()
        loss_b = ((pred_b - y_b) ** 2).mean()

        g_a = flat_grad(loss_a, trunk_params)
        g_b = flat_grad(w_b * loss_b, trunk_params)
        naive = g_a + g_b
        combined = pcgrad_combine(g_a, g_b) if method == "pcgrad" else naive

        opt.zero_grad(set_to_none=False)
        assign_flat(trunk_params, combined)
        for p, g in zip(head_params, torch.autograd.grad(loss_a + w_b * loss_b, head_params)):
            p.grad = g.clone()
        opt.step()

        hist["nmse_a"].append(float(loss_a.detach()) / var_a)
        hist["nmse_b"].append(float(loss_b.detach()) / var_b)
        hist["cos"].append(float(torch.dot(g_a, g_b) / (g_a.norm() * g_b.norm() + 1e-12)))
        hist["ratio"].append(float(g_b.norm() / (g_a.norm() + 1e-12)))
        hist["back_a"].append(bool(torch.dot(g_a, naive) < 0))

    hist = {k: np.array(v) for k, v in hist.items()}
    hist["final_a"] = hist["nmse_a"][-1]
    hist["final_b"] = hist["nmse_b"][-1]
    return hist


def learning_window(hist, tol=0.1):
    """Cosine only means something while the gradients still carry signal."""
    total = hist["nmse_a"] + hist["nmse_b"]
    start, end = total[0], total.min()
    return np.ones_like(total, dtype=bool) if start <= end else total - end > tol * (start - end)

## 5. The conflict, measured

Run the same model on both task pairs and compare the angle. The summary is taken
over the learning window only: once both losses settle, the residual gradients are
numerical noise and their angle means nothing.

In [ ]:
for pair, label in [("conflict", "different features"), ("aligned", "same features")]:
    h = train("naive", pair, seed=0)
    w = learning_window(h)
    print(f"{label:20s} median cos {np.median(h['cos'][w]):+.3f}   "
          f"negative on {(h['cos'][w] < 0).mean() * 100:4.1f}% of learning steps   "
          f"median |g_B|/|g_A| {np.median(h['ratio'][w]):.2f}   "
          f"summed update set task A back on {h['back_a'][w].mean() * 100:4.1f}% of steps")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, pair, label in [(axes[0], "conflict", "Tasks read different features"),
                        (axes[1], "aligned", "Tasks read the same features")]:
    h = train("naive", pair, seed=0)
    w = learning_window(h)
    cut = int(w.sum())
    k = 51
    smooth = np.convolve(h["cos"], np.ones(k) / k, mode="valid")
    ax.axhspan(-1, 0, color="#dc2626", alpha=0.06)
    ax.axvspan(cut, len(h["cos"]), color="black", alpha=0.05, lw=0)
    ax.plot(np.arange(len(smooth)) + k - 1, smooth, color="#2563eb", lw=2)
    ax.axhline(0, color="black", lw=0.9)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("training step")
    ax.set_title(f"{label}\nmedian cos {np.median(h['cos'][w]):+.2f}")
axes[0].set_ylabel("cos(g_A, g_B)")
plt.tight_layout()
plt.show()

## 6. Does the conflict cost anything?

The condition for the summed update to increase task A's loss is

    cos(g_A, g_B) < - ||g_A|| / ||g_B||

so a negative cosine on its own is not enough. With gradients of equal length the
threshold is -1, and no realistic angle clears it. The 2x2 below crosses the two
ingredients: whether the tasks read different features, and whether task B's target
is three times larger.

Three seeds here rather than the eight used in the post, so it finishes quickly.
Expect the same ordering, with slightly different digits.

In [ ]:
SEEDS = [0, 1, 2]
CELLS = [("conflict", 3.0, "conflict + imbalance"),
         ("conflict", 1.0, "conflict only"),
         ("aligned", 3.0, "imbalance only"),
         ("aligned", 1.0, "neither")]

print(f"{'cell':22s} {'naive':>8} {'PCGrad':>8}  {'med cos':>8} {'|gB|/|gA|':>10} {'A back%':>8}")
rows = []
for pair, amp, label in CELLS:
    nv = [train("naive", pair, seed=s, amp_b=amp) for s in SEEDS]
    pc = [train("pcgrad", pair, seed=s, amp_b=amp) for s in SEEDS]
    mean = lambda hs: np.mean([0.5 * (h["final_a"] + h["final_b"]) for h in hs])
    ws = [learning_window(h) for h in nv]
    cos = np.concatenate([h["cos"][w] for h, w in zip(nv, ws)])
    ratio = np.concatenate([h["ratio"][w] for h, w in zip(nv, ws)])
    back = np.concatenate([h["back_a"][w] for h, w in zip(nv, ws)])
    rows.append((label, mean(nv), mean(pc)))
    print(f"{label:22s} {mean(nv):8.4f} {mean(pc):8.4f}  {np.median(cos):+8.3f} "
          f"{np.median(ratio):10.2f} {back.mean() * 100:7.1f}%")

In [ ]:
labels = [r[0] for r in rows]
xs = np.arange(len(rows))
plt.figure(figsize=(8, 4))
plt.bar(xs - 0.19, [r[1] for r in rows], 0.36, color="#dc2626", label="naive sum")
plt.bar(xs + 0.19, [r[2] for r in rows], 0.36, color="#059669", label="PCGrad")
plt.xticks(xs, [l.replace(" + ", "\n+ ").replace(" only", "\nonly") for l in labels], fontsize=9)
plt.ylabel("mean normalised MSE (both tasks)")
plt.title("PCGrad pays in one cell of the four")
plt.legend()
plt.tight_layout()
plt.show()

## 7. The competing baseline

Before reaching for gradient surgery, check what a scalar would have done. Sweep
the weight on task B and see which points on the trade-off a single number can
reach.

In [ ]:
for w_b in [1 / 27, 1 / 9, 1 / 3, 1.0, 3.0]:
    hs = [train("weighted", "conflict", seed=s, weight_b=w_b) for s in SEEDS]
    a = np.mean([h["final_a"] for h in hs])
    b = np.mean([h["final_b"] for h in hs])
    print(f"w_B = {w_b:6.3f}   task A {a:.4f}   task B {b:.4f}   mean {0.5 * (a + b):.4f}")

hs = [train("pcgrad", "conflict", seed=s) for s in SEEDS]
a = np.mean([h["final_a"] for h in hs])
b = np.mean([h["final_b"] for h in hs])
print(f"PCGrad        task A {a:.4f}   task B {b:.4f}   mean {0.5 * (a + b):.4f}")

## Exercises

1. **Move the boundary.** `amp_b` controls the magnitude gap. Sweep it over
   `[1, 1.5, 2, 3, 5]` on the conflicting pair and find the value at which the
   summed update starts moving task A backwards on more than 10% of steps. Compare
   that to where PCGrad's advantage appears.
2. **Widen the trunk.** `HIDDEN` is the contested resource. Raise it to 12 and then
   24. At what width does the conflict stop costing anything, and does the cosine
   change or only the consequence?
3. **Three tasks.** Extend `pcgrad_combine` to the general algorithm: project each
   task's gradient against every other in random order, then sum. Add a third task
   on dimensions 4 and 5 and check whether the pairwise cosines still predict which
   task suffers.
4. **Implement GradNorm's core idea.** Instead of projecting, rescale each task's
   loss each step so the two gradient norms match. Compare it to PCGrad in the
   "imbalance only" cell, where the gradients agree and only the scales differ.
5. **Try TorchJD.** `pip install torchjd` and swap `pcgrad_combine` for its `UPGrad`
   aggregator. The paper argues UPGrad preserves each gradient's influence in
   proportion to its norm; check whether that changes where the run lands on the
   trade-off curve from section 7.